# Classification DPE — modèle final géographique v1

Données corrigées avec les règles de Nouhayla pour Lyon/Pierre-Bénite. Comparaison de trois Random Forest sur 60 000 lignes d'entraînement et 55 235 de validation. Sélection au F1 macro de validation, puis entraînement final sur les 453 680 lignes entraînement + validation. Les 107 666 lignes réservées ont été évaluées après gel des choix. **Ce lot reste interne à des données déjà exploitées, pas externe.**

Ce notebook lit les résultats sauvegardés ; il ne réentraîne rien et ne recalcule pas les prédictions réservées. Les travaux antérieurs sont intégralement conservés dans archives/.


In [1]:
from pathlib import Path
import os
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "MLmodels").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Ouvrez le notebook depuis le dépôt App_Enedis.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.resource_limits import configure
configure()
import pandas as pd
import json
from src.final_model_loader import load_final_bundle
bundle = load_final_bundle()
print("Modèles finaux rechargés ; prédictions de contrôle identiques.")
print("Aucun entraînement ni nouvelle évaluation du lot réservé.")


Modèles finaux rechargés ; prédictions de contrôle identiques.
Aucun entraînement ni nouvelle évaluation du lot réservé.


In [2]:
comparison = bundle["comparison"]
partitions = json.loads((ROOT / "MLmodels/final_geo_v1/partition_distributions.json").read_text(encoding="utf-8"))
display(pd.DataFrame([{ "Partition": name, "Effectif": values["rows"],
                       "Existant": values["sources"].get("existant", 0), "Neuf": values["sources"].get("neuf", 0)}
                     for name, values in partitions.items()]))
display(pd.DataFrame({name: values["classes"] for name, values in partitions.items()}).T)
print("Référence majoritaire de validation :", comparison["baseline_class"])
print("Accuracy :", comparison["baseline_validation"]["accuracy"])
print("F1 macro :", comparison["baseline_validation"]["f1_macro"])
table = pd.DataFrame([{
    "Modèle": m["name"], "Accuracy validation": m["accuracy"], "F1 macro validation": m["f1_macro"],
    "Accuracy existant": m["by_source"]["existant"]["accuracy"],
    "F1 macro existant": m["by_source"]["existant"]["f1_macro"],
    "Accuracy neuf": m["by_source"]["neuf"]["accuracy"],
    "F1 macro neuf": m["by_source"]["neuf"]["f1_macro"],
} for m in comparison["models"]])
display(table)
print("Choix figé sur validation :", comparison["selected_on_validation"])


    Partition  Effectif  Existant    Neuf
0       train    398445    243410  155035
1  validation     55235      4587   50648
2    reserved    107666     57331   50335
3    excluded     54711     12690   42021
                A      B       C      D      E      F      G
train       75108  37459  116835  81069  52418  21764  13792
validation  26067   9491   17081   1643    728    172     53
reserved    29227  13666   39536  16846   6334   1590    467
excluded    25767   7390   14484   4200   2034    558    278
Référence majoritaire de validation : C
Accuracy : 0.3092423282339097
F1 macro : 0.06748555940989151
                  Modèle  Accuracy validation  ...  Accuracy neuf  F1 macro neuf
0    rf_depth10_balanced             0.746646  ...       0.783940       0.316824
1    rf_depth20_balanced             0.747044  ...       0.784138       0.314257
2  rf_depth20_unweighted             0.760460  ...       0.794108       0.309558

[3 rows x 7 columns]
Choix figé sur validation : rf_depth10

In [3]:
rows = []
for model in comparison["models"]:
    for scope, report in [("ensemble", model)] + list(model["by_source"].items()):
        for label in "ABCDEFG":
            rows.append({"Modèle": model["name"], "Population": scope, "DPE": label,
                         **report["report"][label]})
display(pd.DataFrame(rows).pivot(index=["Modèle", "Population"], columns="DPE", values="f1-score"))


DPE                                      A         B  ...         F         G
Modèle                Population                      ...                    
rf_depth10_balanced   ensemble    0.893911  0.542789  ...  0.145110  0.093552
                      existant    0.225000  0.107383  ...  0.148627  0.094508
                      neuf        0.894965  0.551529  ...  0.000000  0.000000
rf_depth20_balanced   ensemble    0.888829  0.526606  ...  0.124352  0.109533
                      existant    0.246914  0.124711  ...  0.126816  0.111340
                      neuf        0.889832  0.535196  ...  0.000000  0.000000
rf_depth20_unweighted ensemble    0.893047  0.464770  ...  0.030000  0.118644
                      existant    0.265060  0.011331  ...  0.032432  0.127273
                      neuf        0.894032  0.474500  ...  0.000000  0.000000

[9 rows x 7 columns]


In [4]:
result = bundle["results"]
print("LOT RÉSERVÉ INTERNE — résultat final déjà évalué, aucun réglage ultérieur")
print("Accuracy :", result["classification"]["accuracy"])
print("F1 macro :", result["classification"]["f1_macro"])
print("Référence majoritaire :", result["baseline"]["classification_majority"])
print("Accuracy référence :", result["baseline"]["classification"]["accuracy"])
print("F1 macro référence :", result["baseline"]["classification"]["f1_macro"])
display(pd.DataFrame({label: result["classification"]["report"][label] for label in "ABCDEFG"}).T)
display(pd.DataFrame([{ "Population": source, "Effectif": v["classification"]["rows"],
                      "Accuracy": v["classification"]["accuracy"], "F1 macro": v["classification"]["f1_macro"]}
                     for source, v in result["by_source"].items()]))


LOT RÉSERVÉ INTERNE — résultat final déjà évalué, aucun réglage ultérieur
Accuracy : 0.554074638233054
F1 macro : 0.40307748057176224
Référence majoritaire : C
Accuracy référence : 0.36720970408485504
F1 macro référence : 0.07673808779772015
   precision    recall  f1-score  support
A   0.939941  0.745920  0.831766  29227.0
B   0.394178  0.542002  0.456419  13666.0
C   0.803615  0.473417  0.595827  39536.0
D   0.356306  0.541375  0.429763  16846.0
E   0.249434  0.313230  0.277716   6334.0
F   0.087729  0.208176  0.123438   1590.0
G   0.058220  0.631692  0.106614    467.0
  Population  Effectif  Accuracy  F1 macro
0   existant     57331  0.387033  0.312649
1       neuf     50335  0.744333  0.304625


In [5]:
per_source = []
for source, values in result["by_source"].items():
    for label in "ABCDEFG":
        per_source.append({"Population": source, "DPE": label, **values["classification"]["report"][label]})
display(pd.DataFrame(per_source))
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt
ConfusionMatrixDisplay(confusion_matrix=__import__("numpy").array(result["classification"]["confusion_matrix"]),
                       display_labels=list("ABCDEFG")).plot()
plt.title("Lot réservé interne — matrice sauvegardée")
plt.show()


   Population DPE  precision    recall  f1-score  support
0    existant   A   0.715090  0.397372  0.510861   1598.0
1    existant   B   0.331946  0.220609  0.265060   4338.0
2    existant   C   0.793823  0.338603  0.474717  26187.0
3    existant   D   0.356306  0.542051  0.429976  16825.0
4    existant   E   0.249434  0.313577  0.277852   6327.0
5    existant   F   0.087729  0.208307  0.123461   1589.0
6    existant   G   0.058220  0.631692  0.106614    467.0
7        neuf   A   0.948893  0.766079  0.847742  27629.0
8        neuf   B   0.405456  0.691467  0.511175   9328.0
9        neuf   C   0.812639  0.737883  0.773459  13349.0
10       neuf   D   0.000000  0.000000  0.000000     21.0
11       neuf   E   0.000000  0.000000  0.000000      7.0
12       neuf   F   0.000000  0.000000  0.000000      1.0
13       neuf   G   0.000000  0.000000  0.000000      0.0


La validation contient 91,7 % de neuf, mais le réservé 46,8 %. Les résultats par population et par classe sont donc nécessaires. La classe A est bien mieux reconnue que F/G ; le modèle produit beaucoup de fausses alertes F/G. Ne pas utiliser ces prédictions comme diagnostic réglementaire. Les anciens scores aléatoires ne sont pas directement comparables à ce protocole. Les filtres de préparation restent appris globalement et BAN n'identifie pas parfaitement les bâtiments : une validation externe reste nécessaire.


In [7]:
print("Variables :", bundle["variables"]["classification"]["features"])
print("Cible :", bundle["variables"]["classification"]["target"])
print("Paramètres :", bundle["variables"]["classification"]["model_parameters"])
print("Modèle : MLmodels/final_geo_v1/classification.joblib")


Variables : ['surface_habitable_logement', 'nombre_niveau_logement', 'age_batiment', 'altitude_moyenne', 'type_energie_principale_chauffage', 'type_batiment', 'zone_climatique']
Cible : etiquette_dpe
Paramètres : {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 20, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': 2, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Modèle : MLmodels/final_geo_v1/classification.joblib
